# 1 - Configuração do ambiente

In [ ]:
import os
from pyspark.sql import SparkSession

def get_spark_session(app_name="Hackathon_Project"):

    os.environ["OCI_IAM_TYPE"] = "resource_principal"


    spark = SparkSession.builder \
        .appName(app_name) \
        .config("spark.driver.memory", "20g") \
        .config("spark.executor.memory", "20g") \
        .config("spark.driver.maxResultSize", "4g") \
        .config("spark.hadoop.fs.oci.client.auth.kind", "resource_principal") \
        .config("spark.hadoop.fs.oci.client.regionCodeOrId", "us-chicago-1") \
        .config("spark.sql.parquet.datetimeRebaseModeInWrite", "LEGACY") \
        .config("spark.sql.parquet.datetimeRebaseModeInRead", "LEGACY") \
        .config("spark.sql.debug.maxToStringFields", "100") \
        .getOrCreate()


    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
    hadoop_conf.set("fs.oci.client.auth.kind", "resource_principal")
    hadoop_conf.set("fs.oci.client.regionCodeOrId", "us-chicago-1")
    hadoop_conf.set("fs.oci.client.custom.authenticator",
                    "com.oracle.bmc.hdfs.auth.ResourcePrincipalsCustomAuthenticator")

    return spark


spark = get_spark_session("Squad_06_Abts")

In [ ]:
from pyspark.sql import SparkSession
import os

def get_max_power_spark(app_name="ABT_Gold_Extreme"):
    # Força a variável de ambiente para garantir
    os.environ["OCI_IAM_TYPE"] = "resource_principal"

    spark = (SparkSession.builder
        .appName(app_name)
        .master("local[10]")
        .config("spark.driver.memory", "38g")
        .config("spark.driver.maxResultSize", "10g")
        .config("spark.sql.shuffle.partitions", "30")
        .config("spark.sql.autoBroadcastJoinThreshold", "100MB")
        # Configurações de Autenticação OCI
        .config("spark.hadoop.fs.oci.client.auth.kind", "resource_principal")
        .config("spark.hadoop.fs.oci.client.regionCodeOrId", "us-chicago-1")
        .getOrCreate())

    # --- O SEGREDO ESTÁ AQUI ---
    # Injeção manual no Hadoop Configuration para evitar o erro de 'null tenantId'
    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
    hadoop_conf.set("fs.oci.client.auth.kind", "resource_principal")
    hadoop_conf.set("fs.oci.client.regionCodeOrId", "us-chicago-1")
    hadoop_conf.set("fs.oci.client.custom.authenticator",
                    "com.oracle.bmc.hdfs.auth.ResourcePrincipalsCustomAuthenticator")

    return spark

# Reinicie o Kernel e execute:
spark = get_max_power_spark()
print("✅ Autenticação Resource Principal injetada com sucesso!")

# 2 - Bibliotecas

In [ ]:
import os
import pandas as pd
from pyspark.sql.functions import col, count, when, isnan, countDistinct, approx_count_distinct, concat, col, lit, substring
from pyspark.sql import functions as F
from pyspark.sql.types import *

# 3 - Funções

In [ ]:
# --------------------------
# Função para retornar o shape
# --------------------------

def get_shape(dataframe):
    linhas = f'Quanitade de linhas: {dataframe.count()}'
    colunas = f'Quanitade de Colunas: {len(dataframe.columns)}'
    return linhas, colunas

In [ ]:
# --------------------------
# Função para retornar tabela agrupada e percentual
# --------------------------
def Freq(pTabela,pColuna, pTop = 1000):

    qtd_total=pTabela.count()

    pTabela.registerTempTable("tab_input")
    frq = spark.sql(
            """
                select
                    {col},
                    count(*) as qtd_absoluto,
                    round(100*(count(*) / {tot}),2) as qtd_percentual
                from
                    tab_input
                group by
                    {col}
                order by
                    2 desc
            """.format(col=pColuna, tot=qtd_total))

    qtd=frq.count()
    print('Quantidade de dominios',qtd)
    if qtd > 500:
        frq.show(pTop,truncate=False)
        return "Dominio muito granular"

    else:
        frq.show(qtd,truncate=False)
        print("volumetria total:",qtd_total)
        return 'Freq da coluna ' + pColuna;

In [ ]:
def comparar_datasets(df1, df2, variavel_alvo):
    """
    Compara o shape e a proporção de uma variável (ex: FPD) entre dois DataFrames.
    """
    # 1. Estatísticas do Primeiro DataFrame
    shape1_rows = df1.count()
    shape1_cols = len(df1.columns)
    # Calcula a média (proporção) da variável alvo
    prop1 = df1.select(F.avg(F.col(variavel_alvo).cast("float"))).collect()[0][0]
    prop1_perc = (prop1 or 0) * 100

    # 2. Estatísticas do Segundo DataFrame
    shape2_rows = df2.count()
    shape2_cols = len(df2.columns)
    prop2 = df2.select(F.avg(F.col(variavel_alvo).cast("float"))).collect()[0][0]
    prop2_perc = (prop2 or 0) * 100

    # 3. Cálculos de Diferença
    diff_rows = shape1_rows - shape2_rows
    perc_perda_rows = (diff_rows / shape1_rows) * 100 if shape1_rows != 0 else 0
    diff_prop = prop1_perc - prop2_perc

    # 4. Exibição dos Resultados
    print(f"{'='*50}")
    print(f" ANÁLISE COMPARATIVA: {variavel_alvo.upper()}")
    print(f"{'='*50}")

    print(f" DATAFRAME 1 (Referência):")
    print(f"   - Shape: ({shape1_rows}, {shape1_cols})")
    print(f"   - Proporção de {variavel_alvo}: {prop1_perc:.2f}%")

    print(f"\n DATAFRAME 2 (Atual):")
    print(f"   - Shape: ({shape2_rows}, {shape2_cols})")
    print(f"   - Proporção de {variavel_alvo}: {prop2_perc:.2f}%")

    print(f"\n{'='*50}")
    print(f" IMPACTO DA TRANSFORMAÇÃO:")
    print(f"   - Linhas removidas: {diff_rows} ({perc_perda_rows:.2f}% de perda)")
    print(f"   - Variação no Target: {diff_prop:.4f} p.p. (pontos percentuais)")
    print(f"{'='*50}")

In [ ]:
# --------------------------
# Função para retornar quantidade e porcentagem NULLs e NaNs
# --------------------------
def ver_nulos(dataframe):
  """
  Retorna um DataFrame Pandas ordenado com a contagem e porcentagem d
  e nulos e NaNs Performance: O(1) Action (apenas um scan na tabela).
  """
  pd.set_option('display.max_rows', 100)
  expressoes = []

  for nome_coluna, tipo_coluna in dataframe.dtypes:
      # Verifica se é float/double para checar também NaN (Not a Number)
      if tipo_coluna in ['double', 'float']:
          condicao = (F.col(nome_coluna).isNull() | isnan(F.col(nome_coluna)))
      else:
          condicao = F.col(nome_coluna).isNull()

      expressoes.append(F.count(F.when(condicao, nome_coluna)).alias(nome_coluna))

  # 2. Executa a action do Spark e converte para Pandas
  resultado = dataframe.select(expressoes).toPandas()

  # 3. Transpõe o resultado para formato de tabela (Colunas viram índices)
  resultado_final = resultado.T.rename(columns={0: 'qtd_nulos'})

  # 4. Calcula a porcentagem diretamente no Pandas
  resultado_final['pct_nulos %'] = round((resultado_final['qtd_nulos'] / dataframe.count()) * 100, 2)

  return resultado_final.sort_values('qtd_nulos', ascending=False)

# 4 - Carregando os Dados

In [ ]:
dfs = {}

In [ ]:
# Tabelas
base_uri = "oci://layer-silver@axshbddfc2lf/"

pastas_silver = [
    'tabela-cadastral',
    'tabela-score-bureau',
    'tabela-telco'
]

print('--- 📂 Iniciando Leitura Direta do Object Storage ---')

for tabela in pastas_silver:
    path_completo = f"{base_uri}{tabela}"
    print(f'📖 Lendo: {tabela}...')

    # O Spark identifica automaticamente as subpastas (partições)
    # de tabela-recarga e tabela-cadastral
    dfs[tabela] = spark.read.parquet(path_completo)

print('\n✅ CARREGAMENTO FINALIZADO')

In [ ]:
# Books
base_uri = "oci://layer-gold@axshbddfc2lf/"

pastas_gold = [
    'book-pagamento',
    'book-recarga'
]

print('--- 📂 Iniciando Leitura Direta do Object Storage ---')

for tabela in pastas_gold:
    path_completo = f"{base_uri}{tabela}"
    print(f'📖 Lendo: {tabela}...')

    # O Spark identifica automaticamente as subpastas (partições)
    # de tabela-recarga e tabela-cadastral
    dfs[tabela] = spark.read.parquet(path_completo)

print('\n✅ CARREGAMENTO FINALIZADO')

In [ ]:
cadastro = dfs['tabela-cadastral']
score = dfs['tabela-score-bureau']
telco = dfs['tabela-telco']
pagamento = dfs['book-pagamento']
recarga = dfs['book-recarga']

# 5 - Score_1 e Score_2

In [ ]:
df = score

In [ ]:
# ----------------------------
# IDENTIFICANDO OOT COM FLAG
#-----------------------------
df = df.withColumn(
    'Flag_OOT',
    F.when(F.col('SAFRA').isin('202502', '202503'), 1).otherwise(0)
)

In [ ]:
# ------------------------------------------------------
# REMOÇÃO DA VARIAVEL PROD POR POSSUIR SOMENTE UM VALOR
#-------------------------------------------------------
df = df.drop('PROD')

In [ ]:
#------------------------------------------
# APLICAÇÃO DE OHE NA VARIÁVEL 'flag_mig2'
#------------------------------------------
df = df.withColumns({
    'MIG_Aquisicao': F.when(F.col('flag_mig2') == 'Aquisição', 1).otherwise(0),
    'MIG_PRE': F.when(F.col('flag_mig2') == 'PRE', 1).otherwise(0),
    'MIG_FLEX': F.when(F.col('flag_mig2') == 'FLEX', 1).otherwise(0),
    'SEM_MIGRACAO': F.when(F.col('flag_mig2') == 'SEM MIGRACAO', 1).otherwise(0)
}).drop('flag_mig2')

In [ ]:
#-------------------------------------------------------------------
# NORMALIZANDO A VARIAVEL 'GRUPO_CONTROLE' PARA O PADRÃO DO DF (int)
#-------------------------------------------------------------------
df = df.withColumn(
    'GRUPO_CONTROLE',
    F.col('GRUPO_CONTROLE').cast('int')
)

In [ ]:
# Padronizando sentinélas para -4: DESCONHECIDO

df = df.withColumns({
    'SCORE_01': F.when(F.col('SCORE_01') == -1, -4).otherwise(F.col('SCORE_01')),
    'SCORE_02': F.when(F.col('SCORE_02') == -1, -4).otherwise(F.col('SCORE_02'))
})

In [ ]:
# Ciração da variavel 'DELTA_SCORE' para captar a diferença entre os Scores

df = df.withColumn(
    'DELTA_SCORE',
    F.when((F.col('SCORE_01') >= 0) & (F.col('SCORE_02') >= 0), F.col('SCORE_01') - F.col('SCORE_02'))
)

In [ ]:
# Como o range de valores é alto indo de negativos a positivos, e ha 15.648 valores naturalmente 0,
# optou-se por criar a sentinéla -9999 para os valores nulos.

df = df.withColumn(
    'DELTA_SCORE',
    F.when(F.col('DELTA_SCORE').isNull(), -9999).otherwise(F.col('DELTA_SCORE'))
)

In [ ]:
# Reordenando dataset
ordem = ['ID_UNICO','GRUPO_CONTROLE', 'Flag_OOT','NUM_CPF', 'SAFRA','FLAG_INSTALACAO',  'MIG_Aquisicao' , 'MIG_PRE', 'MIG_FLEX', 'SEM_MIGRACAO',
         'FPD', 'SCORE_01', 'SCORE_01_MISSING', 'SCORE_02', 'SCORE_02_MISSING', 'DELTA_SCORE']
df = df[ordem]

In [ ]:
#-------------------------------------------------
# REMOVENDO PARA ABT CLIENTES QUE NÃO CONTRATARAM (SEM FPD)
#-------------------------------------------------
df_abt = df.dropna(subset = ['FPD']).drop('FLAG_INSTALACAO')

In [ ]:
# Drop da variavel SEM_MIGRACAO
df_abt = df_abt.drop('SEM_MIGRACAO')

In [ ]:
# Separando Score1 e Score2
score1 = ['ID_UNICO', 'GRUPO_CONTROLE', 'NUM_CPF', 'SAFRA', 'Flag_OOT', 'MIG_Aquisicao', 'MIG_PRE', 'MIG_FLEX', 'FPD', 'SCORE_01', 'SCORE_01_MISSING']

df_score1 = df_abt.select(score1)
df_score2 = df_abt
df_scores = df

In [ ]:
path_gold_score1 = "oci://layer-gold@axshbddfc2lf/score-01"
df_score1.write.mode("overwrite").partitionBy("SAFRA").parquet(path_gold_score1)

path_gold_score2 = "oci://layer-gold@axshbddfc2lf/plus-score-02"
df_score2.write.mode("overwrite").partitionBy("SAFRA").parquet(path_gold_score2)

# 6 Telco

In [ ]:
df = telco

In [ ]:
vars = ['var_26','var_27', 'var_28', 'var_29', 'var_30', 'var_31', 'var_32', 'var_33', 'var_34', 'var_35', 'var_36', 'var_37', 'var_38',
 'var_39', 'var_40', 'var_41', 'var_42', 'var_43', 'var_44', 'var_45', 'var_46', 'var_47', 'var_48', 'var_49', 'var_50', 'var_51', 'var_52',
 'var_53', 'var_54', 'var_55', 'var_56', 'var_57', 'var_58', 'var_59', 'var_60', 'var_61', 'var_62', 'var_63', 'var_64', 'var_65', 'var_66',
 'var_67', 'var_68', 'var_69', 'var_70', 'var_71', 'var_72', 'var_73', 'var_74', 'var_75', 'var_76', 'var_77', 'var_78', 'var_79', 'var_80',
 'var_81', 'var_82', 'var_83', 'var_84', 'var_85', 'var_86', 'var_87', 'var_88', 'var_89', 'var_90', 'var_91', 'var_92', 'var_93']

In [ ]:
#--------------------------------------------------------------
# ALTERANDO PADRÃO DOS VALORES SENTINELAS PARA -4: DESCONHECIDO
#---------------------------------------------------------------
df = df.na.replace(-999, -4, subset=vars)

In [ ]:
df = df.withColumns({
    'anomalia_var_90': F.when(F.col('var_90') > 14000, 1).otherwise(0),
    'var_90': F.when(F.col('var_90') > 14000, -4).otherwise(F.col('var_90'))
})

In [ ]:
#--------------------------------
# APLICANDO OHT NA COLUNA PROD
#--------------------------------
df = df.withColumns({
    'CMV': F.when(F.col('PROD') == 'CMV', 1).otherwise(0),
    'NET': F.when(F.col('PROD') == 'NET', 1).otherwise(0),
    'DTH': F.when(F.col('PROD') == 'DTH', 1).otherwise(0)
}).drop('PROD')

In [ ]:
ordem = ['ID_UNICO', 'NUM_CPF', 'SAFRA', 'CMV', 'NET', 'DTH', 'FPD',
         'var_26', 'var_27', 'var_28', 'var_29', 'var_30', 'var_31', 'var_32', 'var_33', 'var_34', 'var_35', 'var_36', 'var_37', 'var_38', 'var_39', 'var_40',
         'var_41', 'var_42', 'var_43', 'var_44', 'var_45', 'var_46', 'var_47', 'var_48', 'var_49', 'var_50', 'var_51', 'var_52', 'var_53', 'var_54', 'var_55',
         'var_56', 'var_57', 'var_58', 'var_59', 'var_60', 'var_61', 'var_62', 'var_63', 'var_64', 'var_65', 'var_66', 'var_67', 'var_68', 'var_69', 'var_70',
         'var_71', 'var_72', 'var_73', 'var_74', 'var_75', 'var_76', 'var_77','var26_a_var77_MISSING', 'var_78', 'var_79', 'var_80', 'var_81', 'var_82', 'var_83', 'var_84', 'var_85',
         'var_86', 'var_87', 'var_88', 'var_89', 'var_90', 'anomalia_var_90', 'var_91', 'var_92', 'var_93']

In [ ]:
df_telco = df[ordem]

In [ ]:
#--------------------------------------------------------
#REALIZANDO LEFET JOIN DA tabela_telco COM a df_scores
#--------------------------------------------------------

colunas_base = df_scores.columns
colunas_telco = df_telco.columns

colunas_para_trazer = [c for c in colunas_telco if c not in colunas_base or c == 'ID_UNICO']

df_telco_recorte = df_telco.select(colunas_para_trazer)

# Left Join
# Base Esquerda: Score
# Base Direita: Telco
df_join_score_telco = df_scores.join(df_telco_recorte, on = 'ID_UNICO', how = 'left')

vars_novas_geral = [c for c in colunas_para_trazer if c != 'ID_UNICO']

# Regra A: Começa com 'var' -> Imputar -1 'Não se Aplica'
vars_prefixo_var = [c for c in vars_novas_geral if c.startswith('var')]

# Regra B: O restante (Net, DTH, CMV, etc) -> Imputar 0
vars_outras = [c for c in vars_novas_geral if not c.startswith('var')]

# 5. Aplicação dos Fillna
# Aplicar -1 (Não se Aplica) nas variaveis anonimizadas (var_...)
if vars_prefixo_var:
    df_join_score_telco = df_join_score_telco.fillna(-1, subset=vars_prefixo_var)

# Aplicar 0 nas demais numéricas (Produtos, CMV, etc)
if vars_outras:
    df_join_score_telco = df_join_score_telco.fillna(0, subset=vars_outras)

print(f'Shape do df_join_score_telco: {get_shape(df_join_score_telco)}')

In [ ]:
df_join_score_telco = df_join_score_telco.withColumns({
    'var26_a_var77_MISSING' : F.when(F.col('var26_a_var77_MISSING') == -1, 0).otherwise(F.col('var26_a_var77_MISSING')),
    'SEM_TELCO': F.when(F.col('var_92') == -1, 1).otherwise(0)
})

In [ ]:
df_telco_abt = df_join_score_telco.dropna(subset = ['FPD']).drop('FLAG_INSTALACAO')

In [ ]:
df_telco_abt = df_telco_abt.drop('SEM_MIGRACAO', 'NET',  'DTH')

In [ ]:
path_gold_telco = "oci://layer-gold@axshbddfc2lf/plus-telco"
df_telco_abt.write.mode("overwrite").partitionBy("SAFRA").parquet(path_gold_telco)

# 7 - Cadastral

In [ ]:
df = cadastro

In [ ]:
#-------------------------------------------------------------------
# NORMALIZANDO A VARIAVEL 'GRUPO_CONTROLE' PARA O PADRÃO DO DF (int)
#-------------------------------------------------------------------
df = df.withColumn(
    'GRUPO_CONTROLE',
    F.col('GRUPO_CONTROLE').cast('int')
)

In [ ]:
df = df.withColumns({
    'CMV': F.when(F.col('PROD') == 'CMV', 1).otherwise(0),
    'NET': F.when(F.col('PROD') == 'NET', 1).otherwise(0),
    'DTH': F.when(F.col('PROD') == 'DTH', 1).otherwise(0)
}).drop('PROD')

In [ ]:
df = df.withColumns({
    'MIG_Aquisicao': F.when(F.col('flag_mig2') == 'Aquisição', 1).otherwise(0),
    'MIG_PRE': F.when(F.col('flag_mig2') == 'PRE', 1).otherwise(0),
    'MIG_FLEX': F.when(F.col('flag_mig2') == 'FLEX', 1).otherwise(0),
    'SEM_MIGRACAO': F.when(F.col('flag_mig2') == 'SEM MIGRACAO', 1).otherwise(0)
}).drop('flag_mig2')

In [ ]:
df = df.filter(df.STATUSRF != 'NULA')

In [ ]:
df = df.withColumns({
    'STATUSRF_REGULAR': F.when(F.col('STATUSRF') == 'REGULAR', 1).otherwise(0),
    'STATUSRF_PENDENTE_DE_REGULARIZACAO': F.when(F.col('STATUSRF') == 'PENDENTE DE REGULARIZACAO',1).otherwise(0),
    'STATUSRF_SEM_STATUS': F.when(F.col('STATUSRF') == 'SEM_STATUS', 1).otherwise(0),
    'STATUSRF_SUSPENSA': F.when(F.col('STATUSRF') == 'SUSPENSA', 1).otherwise(0),
    'STATUSRF_TITULAR_FALECIDO': F.when(F.col('STATUSRF') == 'TITULAR FALECIDO', 1).otherwise(0),
    'STATUSRF_CANCELADA': F.when(F.col('STATUSRF') == 'CANCELADA', 1).otherwise(0)
})

In [ ]:
df = df.drop('STATUSRF')

In [ ]:
#----------------------------------------------------
# CALCULANDO IDADE DA DATA DE NASCIMENTO ATÉ A SAFRA
# ----------------------------------------------------
coluna_data_safra = F.to_date(F.concat(F.col('SAFRA'), F.lit('01')), 'yyyyMMdd')

df = df.withColumn(
    'IDADE',
    F.floor(
        F.months_between(coluna_data_safra, F.col('DATADENASCIMENTO')) / 12
    ).cast('int')
)

df = df.withColumn(
    'IDADE', F.when(F.col('DATADENASCIMENTO') == '1000-01-01', -4).otherwise(F.col('IDADE'))
).drop('DATADENASCIMENTO')

In [ ]:
df = df.where((F.col('IDADE') < 0) | (F.col('IDADE') >= 18))

In [ ]:
df = df.withColumns({
    'IDADE_AUSENTE': F.when(F.col('IDADE') == -4, 1).otherwise(0),
    'IDADE_ATE_29': F.when((F.col('IDADE') > 0) & (F.col('IDADE') < 30), 1).otherwise(0),
    'IDADE_30_44': F.when((F.col('IDADE') > 29) & (F.col('IDADE') < 45), 1).otherwise(0),
    'IDADE_45_60': F.when((F.col('IDADE') > 44) & (F.col('IDADE') < 61), 1).otherwise(0),
    'IDADE_60_99': F.when((F.col('IDADE') > 60) & (F.col('IDADE') < 100), 1).otherwise(0),
    'IDADE_CENTENARIO': F.when(F.col('IDADE') > 99, 1).otherwise(0)
}).drop('DATADENASCIMENTO_MISSING')

In [ ]:
df = df.withColumns({
    'NUM_STATUSRF_MISSING': F.when(F.col('NUM_STATUSRF') == -1, 1).otherwise(0),
    'NUM_STATUSRF_0': F.when(F.col('NUM_STATUSRF') == 0, 1).otherwise(0),
    'NUM_STATUSRF_1': F.when(F.col('NUM_STATUSRF') == 1, 1).otherwise(0),
    'NUM_STATUSRF_2': F.when(F.col('NUM_STATUSRF') == 2, 1).otherwise(0),
    'NUM_STATUSRF_3': F.when(F.col('NUM_STATUSRF') == 3, 1).otherwise(0),
    'NUM_STATUSRF_4': F.when(F.col('NUM_STATUSRF') == 4, 1).otherwise(0),
    'NUM_STATUSRF_5': F.when(F.col('NUM_STATUSRF') == 5, 1).otherwise(0)
}).drop('NUM_STATUSRF')

In [ ]:
df = df.withColumns({
    'NUM_EMPRE_DIRETOR_1': F.when(F.col('VALOR_EMPR_DIRETOR') == 1.0, 1).otherwise(0),
    'NUM_EMPRE_DIRETOR_2': F.when(F.col('VALOR_EMPR_DIRETOR') == 2.0, 1).otherwise(0),
    'NUM_EMPRE_DIRETOR_3': F.when(F.col('VALOR_EMPR_DIRETOR') == 3.0, 1).otherwise(0),
    'NUM_EMPRE_DIRETOR_4': F.when(F.col('VALOR_EMPR_DIRETOR') == 4.0, 1).otherwise(0),
    'NUM_EMPRE_DIRETOR_5_OU_MAIS': F.when(F.col('VALOR_EMPR_DIRETOR') >= 5.0, 1).otherwise(0)
}).drop('VALOR_EMPR_DIRETOR')

In [ ]:
df = df.withColumns({
    'SAFRA_BF_2020': F.when(F.substring(F.col('SAFRA_BOLSA_FAMILIA').cast('string'), 1, 4) == '2020', 1).otherwise(0),
    'SAFRA_BF_2021': F.when(F.substring(F.col('SAFRA_BOLSA_FAMILIA').cast('string'), 1, 4) == '2021', 1).otherwise(0),
    'SAFRA_BF_2023': F.when(F.substring(F.col('SAFRA_BOLSA_FAMILIA').cast('string'), 1, 4) == '2023', 1).otherwise(0),
    'SAFRA_BF_2024': F.when(F.substring(F.col('SAFRA_BOLSA_FAMILIA').cast('string'), 1, 4) == '2024', 1).otherwise(0)
}).drop('SAFRA_BOLSA_FAMILIA', 'SAFRA_BOLSA_FAMILIA_MISSING')

In [ ]:
ufs_bolsa_familia = [
    'SP', 'RJ', 'BA', 'PE', 'MG', 'PA', 'CE', 'RS', 'GO', 'MA',
    'AM', 'PI', 'PR', 'AL', 'PB', 'RN', 'MT', 'MS', 'DF', 'ES',
    'TO', 'RO', 'SE', 'SC', 'AC', 'AP', 'RR'
]

dicionario_one_hot_uf = {
    f"UF_BF_{uf}": F.when(F.col("UF_BOLSA_FAMILIA") == uf, 1).otherwise(0)
    for uf in ufs_bolsa_familia
}

df = df.withColumns(dicionario_one_hot_uf).drop('UF_BOLSA_FAMILIA')

In [ ]:
df = df.drop('VALOR_EMPR_DIRETOR_MISSING')

In [ ]:
df = df.withColumns({
    # Rendas  Vitalícias
    'FLG_INSS_APOSENTADORIA_PERMANENTE': F.when(F.col('NUMERO_APOSENTADO').isin('41', '42', '32'), 1).otherwise(0),
    'FLG_INSS_PENSAO_MORTE': F.when(F.col('NUMERO_APOSENTADO') == '21', 1).otherwise(0),

    # Rendas Temporárias
    'FLG_INSS_BENEFICIO_TEMPORARIO': F.when(F.col('NUMERO_APOSENTADO').isin('31', '91', '80'), 1).otherwise(0),

    # Rendas Assistenciais / Baixa Renda (1 Salário Mínimo)
    'FLG_INSS_ASSISTENCIAL_BPC_LOAS': F.when(F.col('NUMERO_APOSENTADO').isin('87', '88'), 1).otherwise(0),

    # Qualquer outro código do INSS que não seja -999 nem os mapeados acima
    'FLG_INSS_OUTROS_BENEFICIOS': F.when(
        (~F.col('NUMERO_APOSENTADO').isin('-999', '41', '42', '32', '21', '31', '91', '80', '87', '88')), 1
    ).otherwise(0)
}).drop('NUMERO_APOSENTADO').drop('NUMERO_APOSENTADO_MISSING')

In [ ]:
df = df.drop('MESES_AUXILIO_EMERGENCIAL_MISSING')

In [ ]:
df = df.withColumns({
    'STATUS_FUNC_PRIVADO_ADMITIDO': F.when(F.col('STATUS_FUNC_PRIVADO') == 'ADMITIDO', 1).otherwise(0),
    'STATUS_FUNC_PRIVADO_DISPENSADO': F.when(F.col('STATUS_FUNC_PRIVADO') == 'DISPENSADO', 1).otherwise(0)
}).drop('STATUS_FUNC_PRIVADO')

In [ ]:
# 1. Limpar a data de admissão (tirar o 9999-01-01)
df = df.withColumn(
    'DATA_FUNC_PRIVADO_CLEAN',
    F.when(F.col('DATA_FUNC_PRIVADO') == '9999-01-01', F.lit(None))
     .otherwise(F.to_date(F.col('DATA_FUNC_PRIVADO')))
)

# 2. Transformar a SAFRA em uma Data de Referência
df = df.withColumn(
    'DATA_REFERENCIA_SAFRA',
    F.to_date(F.col('SAFRA').cast('string'), 'yyyyMM')
)

# 3. Calcular o Tempo de Emprego em meses
df = df.withColumn(
    'TEMPO_EMPREGO_MESES',
    F.months_between(F.col('DATA_REFERENCIA_SAFRA'), F.col('DATA_FUNC_PRIVADO_CLEAN'))
)

# 4. Tratamento do Nulo para Machine Learning (Imputando -1 para quem não é CLT)
# Arredondamos para 0 casas decimais para ficar um número inteiro limpo
df = df.withColumn(
    'TEMPO_EMPREGO_MESES',
    F.when(F.col('TEMPO_EMPREGO_MESES').isNull(), -1)
     .otherwise(F.round(F.col('TEMPO_EMPREGO_MESES'), 0))
)

# Agora não dropamos a TEMPO_EMPREGO_MESES!
# Dropamos apenas as datas auxiliares que já cumpriram seu papel.
df = df.drop('DATA_FUNC_PRIVADO_CLEAN', 'DATA_REFERENCIA_SAFRA', 'DATA_FUNC_PRIVADO')

In [ ]:
df = df.withColumns({
    'FLG_EMPREGO_CLT_NOVO_ATE_1_ANO': F.when((F.col('TEMPO_EMPREGO_MESES') >= 0) & (F.col('TEMPO_EMPREGO_MESES') < 12), 1).otherwise(0),
    'FLG_EMPREGO_CLT_1_A_3_ANOS': F.when((F.col('TEMPO_EMPREGO_MESES') >= 12) & (F.col('TEMPO_EMPREGO_MESES') < 36), 1).otherwise(0),
    'FLG_EMPREGO_CLT_3_A_5_ANOS': F.when((F.col('TEMPO_EMPREGO_MESES') >= 36) & (F.col('TEMPO_EMPREGO_MESES') < 60), 1).otherwise(0),
    'FLG_EMPREGO_CLT_MAIS_DE_5_ANOS': F.when(F.col('TEMPO_EMPREGO_MESES') >= 60, 1).otherwise(0)
})

In [ ]:
# 1. Contar a quantidade de fontes/vínculos
# A lógica é: se é SEM_INFORMACAO, é 0.
# Caso contrário, contamos os espaços em branco + 1 para saber quantas palavras/vínculos existem.
df = df.withColumn(
    'QTD_VINCULOS_RENDA',
    F.when(F.col('CONSOLIDADO') == 'SEM_INFORMACAO', 0)
     .otherwise(F.size(F.split(F.col('CONSOLIDADO'), ' ')))
)

# 2. Agora sim, deletamos a coluna de texto consolidada para não sujar o modelo
df = df.drop('CONSOLIDADO')

In [ ]:
df = df.withColumn(
    'var_07_MONETARIO',
    F.round(
        F.when(F.col('var_07_MONETARIO') == -999, -4)
         .otherwise(F.col('var_07_MONETARIO')),
        2
    )
)

In [ ]:
teto_monetario = 9999999.0

df = df.withColumns({
    # 1. Flag valiosa: "Atenção, este cliente tem o perfil anômalo de alto risco (FPD)"
    'FLG_ANOMALIA_VAR_07': F.when(F.col('var_07_MONETARIO') > teto_monetario, 1).otherwise(0),

    # 2. Capping/Teto: Protegendo a variância do modelo
    'var_07_MONETARIO_TRATADA': F.when(F.col('var_07_MONETARIO') > teto_monetario, teto_monetario)
                                 .otherwise(F.col('var_07_MONETARIO'))
})

# Limpando a coluna original cheia de outliers
df = df.drop('var_07_MONETARIO')

In [ ]:
df = df.withColumn(
    'var_02',
    F.when(F.col('var_02') == -999, -4).otherwise(F.col('var_02'))
  )

In [ ]:
df = df.withColumns({
    'var_03': F.when(F.col('var_03') == -1, -4).otherwise(F.col('var_03')),
    'var_05': F.when(F.col('var_05') == -1, -4).otherwise(F.col('var_05'))
})

In [ ]:
df = df.withColumn(
    'CEP_2_DIGITOS',
    F.when(F.col('CEP_3_digitos') == 'XXX', 'XX') # Mantemos o XX para os ausentes
     .otherwise(F.substring(F.col('CEP_3_digitos'), 1, 2))
)

# 2. Extrair a lista de CEPs únicos de 2 dígitos (ação de collect para pegar os domínios)
# Obs: Estamos filtrando o 'XX' para não cair na Armadilha da Variável Dummy!
ceps_unicos = [
    row[0] for row in df.select('CEP_2_DIGITOS').distinct().collect()
    if row[0] != 'XX' and row[0] is not None
]

# 3. Criar as expressões lógicas dinamicamente para cada CEP
dicionario_one_hot_cep = {
    f"FLG_CEP_REGIAO_{cep}": F.when(F.col("CEP_2_DIGITOS") == cep, 1).otherwise(0)
    for cep in ceps_unicos
}

# 4. Aplicar as colunas novas no DataFrame e deletar as colunas de texto antigas
df = df.withColumns(dicionario_one_hot_cep).drop('CEP_3_digitos', 'CEP_2_DIGITOS')

In [ ]:
# 1. Mapeamos todas as colunas que acabamos de criar que começam com "FLG_CEP_REGIAO_"
colunas_regiao = [coluna for coluna in df.columns if coluna.startswith('FLG_CEP_REGIAO_')]

# 2. Criamos uma string com a soma matemática de todas elas
# Vai ficar algo como: "FLG_CEP_REGIAO_69 + FLG_CEP_REGIAO_13 + ..."
expressao_soma = " + ".join(colunas_regiao)

# 3. Recriamos a flag! Se a soma for 0, o CEP não foi informado (era o XXX)
df = df.withColumn(
    'FLG_CEP_NAO_INFORMADO',
    F.when(F.expr(expressao_soma) == 0, 1).otherwise(0)
)

In [ ]:
ordem_final_colunas = [
    # 1. Chaves, Metadados e Target
    'ID_UNICO', 'NUM_CPF', 'SAFRA', 'GRUPO_CONTROLE', 'FLAG_INSTALACAO', 'FPD',

    # 2. Demografia e Situação Cadastral
    'IDADE', 'IDADE_AUSENTE', 'IDADE_ATE_29', 'IDADE_30_44', 'IDADE_45_60', 'IDADE_60_99', 'IDADE_CENTENARIO',
    'STATUSRF_REGULAR', 'STATUSRF_PENDENTE_DE_REGULARIZACAO', 'STATUSRF_SEM_STATUS', 'STATUSRF_SUSPENSA', 'STATUSRF_TITULAR_FALECIDO', 'STATUSRF_CANCELADA',
    'NUM_STATUSRF_MISSING', 'NUM_STATUSRF_0', 'NUM_STATUSRF_1', 'NUM_STATUSRF_2', 'NUM_STATUSRF_3', 'NUM_STATUSRF_4', 'NUM_STATUSRF_5',

    # 3. Perfil de Produto (Telecom)
    'CMV', 'NET', 'DTH', 'MIG_Aquisicao', 'MIG_PRE', 'MIG_FLEX', 'SEM_MIGRACAO',

    # 4. Emprego e Vínculos de Renda (Trabalho Ativo)
    'QTD_VINCULOS_RENDA',
    'FUNC_PRIVADO', 'DATA_FUNC_PRIVADO_MISSING', 'STATUS_FUNC_PRIVADO_ADMITIDO', 'STATUS_FUNC_PRIVADO_DISPENSADO',
    'TEMPO_EMPREGO_MESES', 'FLG_EMPREGO_CLT_NOVO_ATE_1_ANO', 'FLG_EMPREGO_CLT_1_A_3_ANOS', 'FLG_EMPREGO_CLT_3_A_5_ANOS', 'FLG_EMPREGO_CLT_MAIS_DE_5_ANOS',
    'FUNC_PUBL', 'SALARIO_FUNC_PUBL', 'SALARIO_FUNC_PUBL_MISSING',
    'EMPR_DIRETOR', 'NUM_EMPRE_DIRETOR_1', 'NUM_EMPRE_DIRETOR_2', 'NUM_EMPRE_DIRETOR_3', 'NUM_EMPRE_DIRETOR_4', 'NUM_EMPRE_DIRETOR_5_OU_MAIS',

    # 5. INSS e Benefícios Sociais
    'APOSENTADO', 'FLG_INSS_APOSENTADORIA_PERMANENTE', 'FLG_INSS_PENSAO_MORTE', 'FLG_INSS_BENEFICIO_TEMPORARIO', 'FLG_INSS_ASSISTENCIAL_BPC_LOAS', 'FLG_INSS_OUTROS_BENEFICIOS',
    'BOLSA_FAMILIA', 'BENEFICIO_BOLSA_FAMILIA', 'BENEFICIO_BOLSA_FAMILIA_MISSING', 'SAFRA_BF_2020', 'SAFRA_BF_2021', 'SAFRA_BF_2023', 'SAFRA_BF_2024',
    'AUXILLIO_EMERGENCIAL', 'MESES_AUXILIO_EMERGENCIAL',

    # 6. Variáveis de Bureau (Score/Monetário)
    'var_07_MONETARIO_TRATADA', 'FLG_ANOMALIA_VAR_07', 'var_07_MONETARIO_MISSING',
    'var_02', 'var_02_missing', 'var_03', 'var_03_MISSING', 'var_05', 'var_05_MISSING',

    # 7. Geografia - UFs (Em ordem alfabética)
    'UF_BF_AC', 'UF_BF_AL', 'UF_BF_AM', 'UF_BF_AP', 'UF_BF_BA', 'UF_BF_CE', 'UF_BF_DF', 'UF_BF_ES',
    'UF_BF_GO', 'UF_BF_MA', 'UF_BF_MG', 'UF_BF_MS', 'UF_BF_MT', 'UF_BF_PA', 'UF_BF_PB', 'UF_BF_PE',
    'UF_BF_PI', 'UF_BF_PR', 'UF_BF_RJ', 'UF_BF_RN', 'UF_BF_RO', 'UF_BF_RR', 'UF_BF_RS', 'UF_BF_SC',
    'UF_BF_SE', 'UF_BF_SP', 'UF_BF_TO',

    # 8. Geografia - CEPs e Flags de Ausência (Em ordem numérica)
    'FLG_CEP_NAO_INFORMADO',
    'FLG_CEP_REGIAO_00', 'FLG_CEP_REGIAO_01', 'FLG_CEP_REGIAO_02', 'FLG_CEP_REGIAO_03', 'FLG_CEP_REGIAO_04', 'FLG_CEP_REGIAO_05', 'FLG_CEP_REGIAO_06', 'FLG_CEP_REGIAO_07', 'FLG_CEP_REGIAO_08', 'FLG_CEP_REGIAO_09',
    'FLG_CEP_REGIAO_10', 'FLG_CEP_REGIAO_11', 'FLG_CEP_REGIAO_12', 'FLG_CEP_REGIAO_13', 'FLG_CEP_REGIAO_14', 'FLG_CEP_REGIAO_15', 'FLG_CEP_REGIAO_16', 'FLG_CEP_REGIAO_17', 'FLG_CEP_REGIAO_18', 'FLG_CEP_REGIAO_19',
    'FLG_CEP_REGIAO_20', 'FLG_CEP_REGIAO_21', 'FLG_CEP_REGIAO_22', 'FLG_CEP_REGIAO_23', 'FLG_CEP_REGIAO_24', 'FLG_CEP_REGIAO_25', 'FLG_CEP_REGIAO_26', 'FLG_CEP_REGIAO_27', 'FLG_CEP_REGIAO_28', 'FLG_CEP_REGIAO_29',
    'FLG_CEP_REGIAO_30', 'FLG_CEP_REGIAO_31', 'FLG_CEP_REGIAO_32', 'FLG_CEP_REGIAO_33', 'FLG_CEP_REGIAO_34', 'FLG_CEP_REGIAO_35', 'FLG_CEP_REGIAO_36', 'FLG_CEP_REGIAO_37', 'FLG_CEP_REGIAO_38', 'FLG_CEP_REGIAO_39',
    'FLG_CEP_REGIAO_40', 'FLG_CEP_REGIAO_41', 'FLG_CEP_REGIAO_42', 'FLG_CEP_REGIAO_43', 'FLG_CEP_REGIAO_44', 'FLG_CEP_REGIAO_45', 'FLG_CEP_REGIAO_46', 'FLG_CEP_REGIAO_47', 'FLG_CEP_REGIAO_48', 'FLG_CEP_REGIAO_49',
    'FLG_CEP_REGIAO_50', 'FLG_CEP_REGIAO_51', 'FLG_CEP_REGIAO_52', 'FLG_CEP_REGIAO_53', 'FLG_CEP_REGIAO_54', 'FLG_CEP_REGIAO_55', 'FLG_CEP_REGIAO_56', 'FLG_CEP_REGIAO_57', 'FLG_CEP_REGIAO_58', 'FLG_CEP_REGIAO_59',
    'FLG_CEP_REGIAO_60', 'FLG_CEP_REGIAO_61', 'FLG_CEP_REGIAO_62', 'FLG_CEP_REGIAO_63', 'FLG_CEP_REGIAO_64', 'FLG_CEP_REGIAO_65', 'FLG_CEP_REGIAO_66', 'FLG_CEP_REGIAO_67', 'FLG_CEP_REGIAO_68', 'FLG_CEP_REGIAO_69',
    'FLG_CEP_REGIAO_70', 'FLG_CEP_REGIAO_71', 'FLG_CEP_REGIAO_72', 'FLG_CEP_REGIAO_73', 'FLG_CEP_REGIAO_74', 'FLG_CEP_REGIAO_75', 'FLG_CEP_REGIAO_76', 'FLG_CEP_REGIAO_77', 'FLG_CEP_REGIAO_78', 'FLG_CEP_REGIAO_79',
    'FLG_CEP_REGIAO_80', 'FLG_CEP_REGIAO_81', 'FLG_CEP_REGIAO_82', 'FLG_CEP_REGIAO_83', 'FLG_CEP_REGIAO_84', 'FLG_CEP_REGIAO_85', 'FLG_CEP_REGIAO_86', 'FLG_CEP_REGIAO_87', 'FLG_CEP_REGIAO_88', 'FLG_CEP_REGIAO_89',
    'FLG_CEP_REGIAO_90', 'FLG_CEP_REGIAO_91', 'FLG_CEP_REGIAO_92', 'FLG_CEP_REGIAO_93', 'FLG_CEP_REGIAO_94', 'FLG_CEP_REGIAO_95', 'FLG_CEP_REGIAO_96', 'FLG_CEP_REGIAO_97', 'FLG_CEP_REGIAO_98', 'FLG_CEP_REGIAO_99'
]

# Para aplicar a nova ordem na sua ABT final:
df_abt = df.select(*ordem_final_colunas)

In [ ]:
--------------------------------
# LEFT JOIN ENTRE CADASTRAL E ABT
#--------------------------------
cols_cadastro = df_abt.columns
cols_add = df_join_score_telco.columns

# Pegando apenas as colunas que ainda não existem no cadastro (evita duplicidade)
cols_novas = [c for c in cols_add if c not in cols_cadastro or c == 'ID_UNICO']

df_book_cadastro = df_abt.join(
    df_join_score_telco.select(cols_novas),
    on='ID_UNICO',
    how='left'
)

# Criação da Flag de Identificação de Cadastro Ausente no Bureau/Telco
df_book_cadastro = df_book_cadastro.withColumn(
    'SEM_SCORE_E_TELCO',
    F.when(F.col('var_26').isNull(), 1).otherwise(0)
)

# ---------------------------------------------------------
# TRATAMENTO DE NULOS (PÓS-JOIN)
# ---------------------------------------------------------

# 1. Definindo as listas de exceção explícitas
colunas_preencher_zero = ['var26_a_var77_MISSING', 'SEM_TELCO', 'SCORE_01_MISSING', 'SCORE_02_MISSING']
colunas_preencher_menos_9999 = ['DELTA_SCORE']

# 2. Definindo a lista geral (Tudo que é novo, exceto as chaves e as exceções acima)
vars_novas_geral = [
    c for c in cols_novas
    if c not in colunas_preencher_zero
    and c not in colunas_preencher_menos_9999
    and c != 'ID_UNICO'
]

# 3. Aplicando os preenchimentos matemáticos em cascata
df_book_cadastro = df_book_cadastro.fillna(-1, subset=vars_novas_geral)
df_book_cadastro = df_book_cadastro.fillna(0, subset=colunas_preencher_zero)
df_book_cadastro = df_book_cadastro.fillna(-9999, subset=colunas_preencher_menos_9999)

In [ ]:
df_join_cadastro = df_book_cadastro.dropna(subset = ['FPD'])

In [ ]:
df_join_cadastro = df_join_cadastro.drop('FLG_EMPREGO_CLT_1_A_3_ANOS', 'FLG_EMPREGO_CLT_NOVO_ATE_1_ANO', 'SEM_MIGRACAO', 'FLAG_INSTALACAO')

In [ ]:
path_gold_cadastro = "oci://layer-gold@axshbddfc2lf/plus-cadastro"
df_join_cadastro.write.mode("overwrite").partitionBy("SAFRA").parquet(path_gold_cadastro)

# 8 - Recarga

In [ ]:
recarga_join = recarga.drop('NUM_CPF', 'SAFRA')

In [ ]:
df_abt_com_recarga = df_book_cadastro.join(
    recarga_join,
    on='ID_UNICO',
    how='left'
)

In [ ]:
df_abt_com_recarga = df_abt_com_recarga.withColumn(
    'SEM_HISTORICO_RECARGA',
    F.when(F.col('QTD_RECARGAS_HIST').isNull(), 1).otherwise(0)
)

In [ ]:
df_abt_com_recarga = df_abt_com_recarga.fillna(-1, subset = recarga_join.columns)

In [ ]:
df_join_recarga = df_abt_com_recarga.dropna(subset = ['FPD'])

In [ ]:
path_gold_recarga = "oci://layer-gold@axshbddfc2lf/plus-recarga"
df_join_recarga.write.mode("overwrite").partitionBy("SAFRA").parquet(path_gold_recarga)

# 9 - Pagamento (ABT-Final)

In [ ]:
pagamento_join = pagamento.drop('NUM_CPF', 'SAFRA')

In [ ]:
df_abt_com_pagamento = df_abt_com_recarga.join(
    pagamento_join,
    on='ID_UNICO',
    how='left'
)

In [ ]:
df_abt_com_pagamento = df_abt_com_pagamento.withColumn(
    'SEM_HISTORICO_POS_PAGO',
    F.when(F.col('VAL_TOTAL_PAG_FATURA').isNull(), 1).otherwise(0)
)

In [ ]:
df_abt_com_pagamento = df_abt_com_pagamento.fillna(-1, subset = pagamento_join.columns)

In [ ]:
df_join_pagamento = df_abt_com_pagamento.dropna(subset = ['FPD'])

In [ ]:
abt_nao_contratou = df_abt_com_pagamento.filter(F.col('FPD').isNull())

# 10 - Data Quality ABT Final

In [ ]:
get_shape(df_join_pagamento)

In [ ]:
ver_nulos(df_join_pagamento)

In [ ]:
# Unicidade
print(f'Quantidade de registros: {df_join_pagamento.count()}')
print(f'Quantidade de IDs únicos: {df_join_pagamento.select("ID_UNICO").distinct().count()}')

In [ ]:
binarias = [
    'GRUPO_CONTROLE', 'FLAG_INSTALACAO', 'FPD', 'IDADE_AUSENTE', 'IDADE_ATE_29',
    'IDADE_30_44', 'IDADE_45_60', 'IDADE_60_99', 'IDADE_CENTENARIO',
    'STATUSRF_REGULAR', 'STATUSRF_PENDENTE_DE_REGULARIZACAO', 'STATUSRF_SEM_STATUS',
    'STATUSRF_SUSPENSA', 'STATUSRF_TITULAR_FALECIDO', 'STATUSRF_CANCELADA',
    'NUM_STATUSRF_MISSING', 'NUM_STATUSRF_0', 'NUM_STATUSRF_1', 'NUM_STATUSRF_2',
    'NUM_STATUSRF_3', 'NUM_STATUSRF_4', 'NUM_STATUSRF_5', 'CMV', 'NET', 'DTH',
    'MIG_Aquisicao', 'MIG_PRE', 'MIG_FLEX', 'SEM_MIGRACAO', 'FUNC_PRIVADO',
    'DATA_FUNC_PRIVADO_MISSING', 'STATUS_FUNC_PRIVADO_ADMITIDO', 'STATUS_FUNC_PRIVADO_DISPENSADO',
    'FLG_EMPREGO_CLT_NOVO_ATE_1_ANO', 'FLG_EMPREGO_CLT_1_A_3_ANOS', 'FLG_EMPREGO_CLT_3_A_5_ANOS',
    'FLG_EMPREGO_CLT_MAIS_DE_5_ANOS', 'FUNC_PUBL', 'SALARIO_FUNC_PUBL_MISSING',
    'EMPR_DIRETOR', 'NUM_EMPRE_DIRETOR_1', 'NUM_EMPRE_DIRETOR_2', 'NUM_EMPRE_DIRETOR_3',
    'NUM_EMPRE_DIRETOR_4', 'NUM_EMPRE_DIRETOR_5_OU_MAIS', 'APOSENTADO',
    'FLG_INSS_APOSENTADORIA_PERMANENTE', 'FLG_INSS_PENSAO_MORTE', 'FLG_INSS_BENEFICIO_TEMPORARIO',
    'FLG_INSS_ASSISTENCIAL_BPC_LOAS', 'FLG_INSS_OUTROS_BENEFICIOS', 'BOLSA_FAMILIA',
    'BENEFICIO_BOLSA_FAMILIA_MISSING', 'SAFRA_BF_2020', 'SAFRA_BF_2021', 'SAFRA_BF_2023',
    'SAFRA_BF_2024', 'AUXILLIO_EMERGENCIAL', 'MESES_AUXILIO_EMERGENCIAL_MISSING',
    'FLG_ANOMALIA_VAR_07', 'var_07_MONETARIO_MISSING', 'var_02_missing', 'var_03_MISSING',
    'var_05_MISSING', 'UF_BF_AC', 'UF_BF_AL', 'UF_BF_AM', 'UF_BF_AP', 'UF_BF_BA',
    'UF_BF_CE', 'UF_BF_DF', 'UF_BF_ES', 'UF_BF_GO', 'UF_BF_MA', 'UF_BF_MG', 'UF_BF_MS',
    'UF_BF_MT', 'UF_BF_PA', 'UF_BF_PB', 'UF_BF_PE', 'UF_BF_PI', 'UF_BF_PR', 'UF_BF_RJ',
    'UF_BF_RN', 'UF_BF_RO', 'UF_BF_RR', 'UF_BF_RS', 'UF_BF_SC', 'UF_BF_SE', 'UF_BF_SP',
    'UF_BF_TO', 'FLG_CEP_NAO_INFORMADO', 'FLG_CEP_REGIAO_00', 'FLG_CEP_REGIAO_01',
    'FLG_CEP_REGIAO_02', 'FLG_CEP_REGIAO_03', 'FLG_CEP_REGIAO_04', 'FLG_CEP_REGIAO_05',
    'FLG_CEP_REGIAO_06', 'FLG_CEP_REGIAO_07', 'FLG_CEP_REGIAO_08', 'FLG_CEP_REGIAO_09',
    'FLG_CEP_REGIAO_10', 'FLG_CEP_REGIAO_11', 'FLG_CEP_REGIAO_12', 'FLG_CEP_REGIAO_13',
    'FLG_CEP_REGIAO_14', 'FLG_CEP_REGIAO_15', 'FLG_CEP_REGIAO_16', 'FLG_CEP_REGIAO_17',
    'FLG_CEP_REGIAO_18', 'FLG_CEP_REGIAO_19', 'FLG_CEP_REGIAO_20', 'FLG_CEP_REGIAO_21',
    'FLG_CEP_REGIAO_22', 'FLG_CEP_REGIAO_23', 'FLG_CEP_REGIAO_24', 'FLG_CEP_REGIAO_25',
    'FLG_CEP_REGIAO_26', 'FLG_CEP_REGIAO_27', 'FLG_CEP_REGIAO_28', 'FLG_CEP_REGIAO_29',
    'FLG_CEP_REGIAO_30', 'FLG_CEP_REGIAO_31', 'FLG_CEP_REGIAO_32', 'FLG_CEP_REGIAO_33',
    'FLG_CEP_REGIAO_34', 'FLG_CEP_REGIAO_35', 'FLG_CEP_REGIAO_36', 'FLG_CEP_REGIAO_37',
    'FLG_CEP_REGIAO_38', 'FLG_CEP_REGIAO_39', 'FLG_CEP_REGIAO_40', 'FLG_CEP_REGIAO_41',
    'FLG_CEP_REGIAO_42', 'FLG_CEP_REGIAO_43', 'FLG_CEP_REGIAO_44', 'FLG_CEP_REGIAO_45',
    'FLG_CEP_REGIAO_46', 'FLG_CEP_REGIAO_47', 'FLG_CEP_REGIAO_48', 'FLG_CEP_REGIAO_49',
    'FLG_CEP_REGIAO_50', 'FLG_CEP_REGIAO_51', 'FLG_CEP_REGIAO_52', 'FLG_CEP_REGIAO_53',
    'FLG_CEP_REGIAO_54', 'FLG_CEP_REGIAO_55', 'FLG_CEP_REGIAO_56', 'FLG_CEP_REGIAO_57',
    'FLG_CEP_REGIAO_58', 'FLG_CEP_REGIAO_59', 'FLG_CEP_REGIAO_60', 'FLG_CEP_REGIAO_61',
    'FLG_CEP_REGIAO_62', 'FLG_CEP_REGIAO_63', 'FLG_CEP_REGIAO_64', 'FLG_CEP_REGIAO_65',
    'FLG_CEP_REGIAO_66', 'FLG_CEP_REGIAO_67', 'FLG_CEP_REGIAO_68', 'FLG_CEP_REGIAO_69',
    'FLG_CEP_REGIAO_70', 'FLG_CEP_REGIAO_71', 'FLG_CEP_REGIAO_72', 'FLG_CEP_REGIAO_73',
    'FLG_CEP_REGIAO_74', 'FLG_CEP_REGIAO_75', 'FLG_CEP_REGIAO_76', 'FLG_CEP_REGIAO_77',
    'FLG_CEP_REGIAO_78', 'FLG_CEP_REGIAO_79', 'FLG_CEP_REGIAO_80', 'FLG_CEP_REGIAO_81',
    'FLG_CEP_REGIAO_82', 'FLG_CEP_REGIAO_83', 'FLG_CEP_REGIAO_84', 'FLG_CEP_REGIAO_85',
    'FLG_CEP_REGIAO_86', 'FLG_CEP_REGIAO_87', 'FLG_CEP_REGIAO_88', 'FLG_CEP_REGIAO_89',
    'FLG_CEP_REGIAO_90', 'FLG_CEP_REGIAO_91', 'FLG_CEP_REGIAO_92', 'FLG_CEP_REGIAO_93',
    'FLG_CEP_REGIAO_94', 'FLG_CEP_REGIAO_95', 'FLG_CEP_REGIAO_96', 'FLG_CEP_REGIAO_97',
    'FLG_CEP_REGIAO_98', 'FLG_CEP_REGIAO_99', 'Flag_OOT', 'SCORE_01_MISSING',
    'SCORE_02_MISSING', 'var26_a_var77_MISSING', 'anomalia_var_90', 'SEM_TELCO',
    'SEM_SCORE_E_TELCO', 'SEM_HISTORICO_RECARGA', 'SEM_HISTORICO_POS_PAGO',
    'FLG_REGIONAL_1', 'FLG_REGIONAL_10', 'FLG_REGIONAL_2', 'FLG_REGIONAL_3',
    'FLG_REGIONAL_4', 'FLG_REGIONAL_5', 'FLG_REGIONAL_6', 'FLG_REGIONAL_7',
    'FLG_REGIONAL_8', 'FLG_REGIONAL_9'
]

numericas = [
    'IDADE', 'QTD_VINCULOS_RENDA', 'TEMPO_EMPREGO_MESES', 'SALARIO_FUNC_PUBL',
    'BENEFICIO_BOLSA_FAMILIA', 'MESES_AUXILIO_EMERGENCIAL', 'var_07_MONETARIO_TRATADA',
    'var_02', 'var_03', 'var_05', 'SCORE_01', 'SCORE_02', 'DELTA_SCORE', 'var_26',
    'var_27', 'var_28', 'var_29', 'var_30', 'var_31', 'var_32', 'var_33', 'var_34',
    'var_35', 'var_36', 'var_37', 'var_38', 'var_39', 'var_40', 'var_41', 'var_42',
    'var_43', 'var_44', 'var_45', 'var_46', 'var_47', 'var_48', 'var_49', 'var_50',
    'var_51', 'var_52', 'var_53', 'var_54', 'var_55', 'var_56', 'var_57', 'var_58',
    'var_59', 'var_60', 'var_61', 'var_62', 'var_63', 'var_64', 'var_65', 'var_66',
    'var_67', 'var_68', 'var_69', 'var_70', 'var_71', 'var_72', 'var_73', 'var_74',
    'var_75', 'var_76', 'var_77', 'var_78', 'var_79', 'var_80', 'var_81', 'var_82',
    'var_83', 'var_84', 'var_85', 'var_86', 'var_87', 'var_88', 'var_89', 'var_90',
    'var_91', 'var_92', 'var_93', 'QTD_DIAS_ULTIMA_RECARGA', 'QTD_RECARGAS_HIST',
    'QTD_CELULARES_RECARREGADOS', 'QTD_RECARGAS_U30D', 'QTD_RECARGAS_U31_90D',
    'VAL_TOTAL_CREDITO_HIST', 'VAL_MAX_RECARGA', 'VAL_MEDIA_RECARGA',
    'VAL_TOTAL_REAL_HIST', 'VAL_TICKET_MEDIO_REAL', 'QTD_USO_SOS_HIST',
    'VAL_TOTAL_SOS_TOMADO', 'VAL_TOTAL_TAXA_SOS_PAGA', 'VAL_MEDIA_SOS_TOMADO',
    'VAL_REAL_U30D', 'VAL_REAL_U31_90D', 'VAL_MEDIA_DIAS_ENTRE_RECARGAS',
    'PCT_MOMENTUM_RECARGA_30D', 'PCT_DEPENDENCIA_SOS', 'QTD_TURNO_MADRUGADA',
    'QTD_TURNO_MANHA', 'QTD_TURNO_NOITE', 'QTD_TURNO_TARDE', 'QTD_CANAL_BANCOS_TRADICIONAL',
    'QTD_CANAL_FINTECH_OU_CARTEIRA_DIGITAL', 'QTD_CANAL_GATEWAYS_CAMPANHAS',
    'QTD_CANAL_NAO_DECLARADO', 'QTD_CANAL_NAO_INFORMADO', 'QTD_CANAL_OUTROS',
    'QTD_CANAL_VAREJO_F_SICO_DINHEIRO_VIVO', 'QTD_PLANO_DESCONHECIDO',
    'QTD_PLANO_NICHO_OUTROS', 'QTD_PLANO_PLANO_CONTROLE', 'QTD_PLANO_PRE_PAGO_CORE',
    'QTD_PLANO_PRE_PAGO_LEGADO', 'QTD_PLANO_PRE_PAGO_REGIONAL_CAMPANHA',
    'QTD_CARTAO_AX', 'QTD_CARTAO_FV', 'QTD_CARTAO_FW', 'QTD_CARTAO_G6', 'QTD_CARTAO_I4',
    'QTD_CARTAO_I8', 'QTD_CARTAO_IA', 'QTD_CARTAO_IB', 'QTD_CARTAO_IC', 'QTD_CARTAO_IW',
    'QTD_CARTAO_N9', 'QTD_CARTAO_NAO_DETERMINADO', 'QTD_CARTAO_OR', 'QTD_CARTAO_OUTROS',
    'QTD_CARTAO_PY', 'QTD_CARTAO_PZ', 'QTD_CARTAO_UB', 'QTD_CARTAO_UC', 'QTD_CARTAO_UD',
    'QTD_CARTAO_UE', 'QTD_CARTAO_WT', 'QTD_ORIGEM_ATIVPROMOCAO', 'QTD_ORIGEM_CHIPPRE_R_30',
    'QTD_ORIGEM_FORCAZB2', 'QTD_ORIGEM_NAO_DETERMINADO', 'QTD_ORIGEM_NAOSEAPLICA',
    'QTD_ORIGEM_OUTROS', 'QTD_ORIGEM_REC_ONLINE', 'QTD_PLATAFORMA_CONTROLE',
    'QTD_PLATAFORMA_CONTROLE_FACIL', 'QTD_PLATAFORMA_MACHINE_TO_MACHINE_SPECIAL',
    'QTD_PLATAFORMA_PARCEIROS_MVNO_DIGITAL', 'QTD_PLATAFORMA_PR_PAGO',
    'QTD_PLATAFORMA_PR_PAGO_BANDA_LARGA', 'QTD_PLATAFORMA_PR_PAGO_CHIP_DE_NICHO',
    'QTD_PLATAFORMA_PR_PAGO_FLEX_DIGITAL', 'QTD_PLATAFORMA_P_S_PAGO',
    'QTD_PLATAFORMA_P_S_PAGO_BANDA_LARGA', 'QTD_PLATAFORMA_P_S_PAGO_RIO_APPLE_WATCH',
    'QTD_PLATAFORMA_SISTEMAS_IOT', 'QTD_PLATAFORMA_TELEMETRIA',
    'QTD_TIPO_RECAR_ADICIONAL_DO_CLARO_CONTROLE', 'QTD_TIPO_RECAR_DESCONHECIDO',
    'QTD_TIPO_RECAR_FRANQUIA_DO_CLARO_CONTROLE', 'QTD_TIPO_RECAR_N_O_DETERMINADO',
    'VAL_TOTAL_PAG_FATURA', 'VAL_TOTAL_JUROS_PAGOS', 'QTD_MAX_DIAS_ATRASO',
    'QTD_TOTAL_FATURAS_PAGAS', 'QTD_STAT_FAT_C', 'QTD_STAT_FAT_O',
    'QTD_FORMA_PAG_ACORDO_PAGAMENTO', 'QTD_FORMA_PAG_ARRECADACAO_BANCARIA',
    'QTD_FORMA_PAG_DEBITO_DIRETO', 'QTD_FORMA_PAG_ONLINE', 'QTD_TIPO_PAG_30001',
    'QTD_TIPO_PAG_30003', 'QTD_TIPO_PAG_30006', 'QTD_TIPO_PAG_30007',
    'QTD_BANCO_FINTECH', 'QTD_BANCO_NAO_INFORMADO', 'QTD_BANCO_NT1', 'QTD_BANCO_OUTROS',
    'QTD_BANCO_TRADICIONAL', 'QTD_STAT_PAG_B', 'QTD_STAT_PAG_C',
    'QTD_STAT_PAG_DESCONHECIDO', 'QTD_STAT_PAG_P', 'QTD_STAT_PAG_R', 'QTD_ALOC_CRT',
    'QTD_ALOC_CRTW', 'QTD_ALOC_DESCONHECIDO', 'QTD_ALOC_OUTROS', 'QTD_ALOC_PYM',
    'QTD_PERFIL_ATRASO_ADIANTADO', 'QTD_PERFIL_ATRASO_ATRASADO', 'QTD_PERFIL_ATRASO_EM_DIA',
    'QTD_PERFIL_ATRASO_SEM_STATUS', 'QTD_FREQ_JUROS_ALTO', 'QTD_FREQ_JUROS_INSIGNIFICANTE',
    'QTD_FREQ_JUROS_MODERADO', 'QTD_FREQ_JUROS_MUITO_ALTO', 'QTD_FREQ_JUROS_SEM_JUROS'
]

In [ ]:
for c in binarias[0:50]:
    print(f'Variavel: {c}')
    Freq(df_join_pagamento, c)
    print('\n')

In [ ]:
for c in binarias[50:100]:
    print(f'Variavel: {c}')
    Freq(df_join_pagamento, c)
    print('\n')

In [ ]:
for c in binarias[100:150]:
    print(f'Variavel: {c}')
    Freq(df_join_pagamento, c)
    print('\n')

In [ ]:
for c in binarias[150:]:
    print(f'Variavel: {c}')
    Freq(df_join_pagamento, c)
    print('\n')

In [ ]:
df_join_pagamento.select(numericas[0:50]).summary().show()

In [ ]:
df_join_pagamento.select(numericas[50:100]).summary().show()

In [ ]:
df_join_pagamento.select(numericas[100:150]).summary().show()

In [ ]:
df_join_pagamento.select(numericas[150:]).summary().show()

In [ ]:
 comparar_datasets(df_score1, df_join_pagamento, 'FPD')

## Correções

In [ ]:
cadastral = spark.read.parquet("oci://layer-gold@axshbddfc2lf/plus-cadastro")
recarga = spark.read.parquet("oci://layer-gold@axshbddfc2lf/plus-recarga")

In [ ]:
df_join_pagamento = df_join_pagamento.withColumn('SALARIO_FUNC_PUBL', F.when(F.col('SALARIO_FUNC_PUBL') < -1, 0).otherwise(F.col('SALARIO_FUNC_PUBL')))
cadastral = cadastral.withColumn('SALARIO_FUNC_PUBL', F.when(F.col('SALARIO_FUNC_PUBL') < -1, 0).otherwise(F.col('SALARIO_FUNC_PUBL')))
recarga = recarga.withColumn('SALARIO_FUNC_PUBL', F.when(F.col('SALARIO_FUNC_PUBL') < -1, 0).otherwise(F.col('SALARIO_FUNC_PUBL')))

In [ ]:
cols_recarca = ['VAL_TOTAL_REAL_HIST', 'VAL_TICKET_MEDIO_REAL', 'VAL_REAL_U30D', 'VAL_REAL_U31_90D', 'PCT_MOMENTUM_RECARGA_30D']

for variavel in cols_recarca:
    recarga = recarga.withColumn(variavel, F.when((F.col(variavel) < 0) & (F.col('SEM_HISTORICO_RECARGA') != 1), 0).otherwise(F.col(variavel)))

for variavel in cols_recarca:
    df_join_pagamento = df_join_pagamento.withColumn(variavel, F.when((F.col(variavel) < 0) & (F.col('SEM_HISTORICO_RECARGA') != 1), 0).otherwise(F.col(variavel)))

Salvamento Final

In [ ]:
path_gold_cadastro = "oci://layer-gold@axshbddfc2lf/plus-cadastro"
cadastral.write.mode("overwrite").partitionBy("SAFRA").parquet(path_gold_cadastro)

In [ ]:
path_gold_recarga = "oci://layer-gold@axshbddfc2lf/plus-recarga"
recarga.write.mode("overwrite").partitionBy("SAFRA").parquet(path_gold_recarga)

In [ ]:
path_gold_pagamento = "oci://layer-gold@axshbddfc2lf/ABT-final"
df_join_pagamento.write.mode("overwrite").partitionBy("SAFRA").parquet(path_gold_pagamento)